# Logistic Regression Model for Port Congestion Risk

## Objective

The objective of this stage is to develop a predictive model capable of identifying congestion risk within the port logistics system.

Logistic regression is selected due to its interpretability and suitability for small datasets. The model estimates the probability that the port system enters a congestion state based on operational signals derived during the feature engineering stage.

Two operational feature groups identified during feature selection are evaluated:

Group A – Import evacuation pressure  
Group B – Export yard pressure

These feature groups represent different operational dynamics that may influence congestion risk. Model performance will be compared to determine which group better explains congestion conditions within the port system.

## Import Libraries

In [4]:
import pandas as pd
import numpy as np

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

from sklearn.model_selection import LeaveOneOut
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## Dataset Input

The dataset used in this notebook is the feature-engineered dataset produced in the previous stage of the analysis pipeline. This dataset contains operational variables and engineered predictors along with the congestion risk indicator.

In [5]:
df_f = pd.read_csv("../../data/JNPA_feature_engineered.csv")
df_f.columns


Index(['Month-Year', 'Imp_Dwell_Overall', 'Imp_Dwell_Truck', 'Imp_Dwell_Train',
       'Parking_Dwell', 'Imp_Transit_CFS', 'Imp_Transit_ICD',
       'Exp_Dwell_Overall', 'Exp_Dwell_Truck', 'Exp_Dwell_Train',
       'Exp_Transit_CFS', 'Exp_Transit_ICD', 'Total_TEUs_Handled', 'Date',
       'Volume_Lag1', 'Rail_Friction', 'Is_Monsoon', 'Stress', 'Risk'],
      dtype='object')

## Target Variable

The target variable represents congestion risk within the port system. It is derived from import dwell time conditions and identifies months with elevated congestion levels.

In [6]:
y = df_f["Risk"]

## Feature Groups

Based on the feature selection stage, two operational feature groups are evaluated.

Group A represents import evacuation pressure within the port system.

Group B captures export yard congestion pressure which may indirectly affect import container evacuation.

In [7]:
features_A = ["Volume_Lag1","Rail_Friction","Stress"]
X_A = df_f[features_A]

features_B = ["Volume_Lag1","Rail_Friction","Parking_Dwell"]
X_B = df_f[features_B]



## Model Pipeline

Feature scaling is applied using StandardScaler to normalize predictor values before model training. Scaling is implemented within a pipeline to ensure that normalization occurs separately within each cross-validation fold, preventing information leakage.

Logistic regression is used as the classification model, with balanced class weights applied to address class imbalance in the congestion risk variable.

In [8]:
pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(class_weight="balanced"))
])

## Cross Validation

Due to the limited dataset size, Leave-One-Out Cross Validation (LOOCV) is used to evaluate model performance.

LOOCV trains the model on all observations except one and tests on the remaining observation. This process is repeated for each observation in the dataset, ensuring that every data point is used as a test sample once.

In [9]:
loo = LeaveOneOut()

### Model Training – Feature Group A

In [10]:
preds_A = []
true_A = []

for train, test in loo.split(X_A):

    X_train, X_test = X_A.iloc[train], X_A.iloc[test]
    y_train, y_test = y.iloc[train], y.iloc[test]

    pipeline.fit(X_train, y_train)

    p = pipeline.predict(X_test)

    preds_A.append(p[0])
    true_A.append(y_test.values[0])

### Model Performance – Feature Group A

In [11]:
print("Accuracy:", accuracy_score(true_A, preds_A))

print("\nConfusion Matrix")
print(confusion_matrix(true_A, preds_A))

print("\nClassification Report")
print(classification_report(true_A, preds_A))

Accuracy: 0.7352941176470589

Confusion Matrix
[[19  6]
 [ 3  6]]

Classification Report
              precision    recall  f1-score   support

           0       0.86      0.76      0.81        25
           1       0.50      0.67      0.57         9

    accuracy                           0.74        34
   macro avg       0.68      0.71      0.69        34
weighted avg       0.77      0.74      0.75        34



### Model Training – Feature Group B

In [12]:
preds_B = []
true_B = []

for train, test in loo.split(X_B):

    X_train, X_test = X_B.iloc[train], X_B.iloc[test]
    y_train, y_test = y.iloc[train], y.iloc[test]

    pipeline.fit(X_train, y_train)

    p = pipeline.predict(X_test)

    preds_B.append(p[0])
    true_B.append(y_test.values[0])

### Model Performance – Feature Group B

In [13]:
print("Accuracy:", accuracy_score(true_B, preds_B))

print("\nConfusion Matrix")
print(confusion_matrix(true_B, preds_B))

print("\nClassification Report")
print(classification_report(true_B, preds_B))

Accuracy: 0.7352941176470589

Confusion Matrix
[[19  6]
 [ 3  6]]

Classification Report
              precision    recall  f1-score   support

           0       0.86      0.76      0.81        25
           1       0.50      0.67      0.57         9

    accuracy                           0.74        34
   macro avg       0.68      0.71      0.69        34
weighted avg       0.77      0.74      0.75        34



## Model Comparison

The two feature groups were evaluated to determine which operational dynamics better explain congestion risk.

Feature Group A focuses on import evacuation conditions, while Feature Group B captures export yard congestion pressure.

The model with stronger detection of congestion months while maintaining operational interpretability is selected as the primary diagnostic model.